# 19 · The parser: C table against Python table (paper §9)
Code: `VAPMIP/PtolC/zork.c` (`VERBS[]`, `zork_parse`), `VAPMIP/zork_parser.py` (`OPERATOR_VERBS`, `VERB_TO_OP`, `OP_NAMES`). Exhaustive comparison.

In [1]:
from pn import *
L = Ledger("19 · parser tables")
import re
with quiet(): import zork_parser as zp
c = open(os.path.join(VAPMIP, "PtolC", "zork.c")).read(); tbl = c[c.index("VerbEntry VERBS[]"):]; tbl = tbl[:tbl.index("};")]
cv = {m.group(1): int(m.group(2)) for m in re.finditer(r'\{\s*"([^"]+)"\s*,\s*(\d+)\s*\}', tbl)}
pv = dict(zp.VERB_TO_OP)
print(len(cv), "verbs in the C table;", len(pv), "in the Python table")
print("operators:", {k: v for k, v in zp.OP_NAMES.items()})

197 verbs in the C table; 207 in the Python table
operators: {0: 'identity', 1: 'negate', 2: 'bind', 3: 'name', 4: 'apply', 5: 'abstract', 6: 'branch', 7: 'iterate', 8: 'recurse', 9: 'allocate', 10: 'query', 11: 'dereference', 12: 'compose', 13: 'parallelize', 14: 'interrupt', 15: 'emit'}


In [2]:
only_py = sorted(set(pv) - set(cv)); only_c = sorted(set(cv) - set(pv)); diff = sorted(v for v in set(cv) & set(pv) if cv[v] != pv[v])
print("only in Python:", only_py); print("only in C:", only_c); print("same verb, different operator:", [(v, cv[v], pv[v]) for v in diff])
L.check("both tables cover all 16 operators", len(set(cv.values())) == 16 and len(set(pv.values())) == 16)
L.check("every compiled verb is in the Python table", not only_c, str(only_c))
L.check("zork.h: the C table 'mirrors zork_parser.py exactly' (no verb maps to a different operator)", not diff, str([(v, cv[v], pv[v]) for v in diff]))
L.check("zork.h: same verb table (the Python table has no extra verbs)", not only_py, str(only_py))
multi = {}
for op, vs in zp.OPERATOR_VERBS.items():
    for v in vs: multi.setdefault(v, []).append(op)
multi = {v: o for v, o in multi.items() if len(set(o)) > 1}
print("verbs listed under more than one operator in Python:", multi)
agree = sorted(v for v in multi if cv.get(v) == pv[v])
L.check("cause: Python lists 6 verbs under two operators and keeps the last; the C table lists each once; 5 of the 6 then differ", len(multi) == 6 and sorted(set(multi) - set(agree)) == sorted(diff), f"doubly listed: {sorted(multi)}; agree: {agree}; differ: {diff}")

only in Python: ['again', 'and also', 'break open', 'g', 'look for', 'these', 'those', 'type', 'with', 'z']
only in C: []
same verb, different operator: [('break', 1, 14), ('compose', 9, 12), ('print', 12, 15), ('scan', 0, 7), ('write', 9, 12)]
verbs listed under more than one operator in Python: {'scan': [0, 7], 'break': [1, 14], 'ask': [3, 10], 'write': [9, 12], 'compose': [9, 12], 'print': [12, 15]}


True

## One line through the compiled REPL

In [3]:
r = subprocess.run([PTOLEMY, "-r", "-n"], input="what is a tree\n\n", capture_output=True, text=True, timeout=60, cwd=os.path.dirname(PTOLEMY))
z = [l for l in (r.stdout + r.stderr).splitlines() if l.startswith("[zork]")]
print(z)
L.check("`what is a tree` parses to operator e0/identity with prompt `identity is tree`", any("e0/identity" in l and "identity is tree" in l for l in z))

['[zork] e0/identity  "identity is tree"']


True

## Ledger

In [4]:
_n = L.show()

LEDGER — 19 · parser tables
  [PASS ] both tables cover all 16 operators
  [PASS ] every compiled verb is in the Python table
          []
  [FAIL ] zork.h: the C table 'mirrors zork_parser.py exactly' (no verb maps to a different operator)
          [('break', 1, 14), ('compose', 9, 12), ('print', 12, 15), ('scan', 0, 7), ('write', 9, 12)]
  [FAIL ] zork.h: same verb table (the Python table has no extra verbs)
          ['again', 'and also', 'break open', 'g', 'look for', 'these', 'those', 'type', 'with', 'z']
  [PASS ] cause: Python lists 6 verbs under two operators and keeps the last; the C table lists each once; 5 of the 6 then differ
          doubly listed: ['ask', 'break', 'compose', 'print', 'scan', 'write']; agree: ['ask']; differ: ['break', 'compose', 'print', 'scan', 'write']
  [PASS ] `what is a tree` parses to operator e0/identity with prompt `identity is tree`
  -> PASS 4 · FAIL 2 · PROSE 0
